# 01 — Preprocessing: German Credit (Statlog)

In [ ]:
import pandas as pd
import numpy as np
import os

RAW_PATH  = '../data/raw/south_german/german.data'
OUT_PATH  = '../data/processed/v4/south_german_credit.csv'

os.makedirs('../data/processed/v4', exist_ok=True)

## 1. Load Raw Data

In [ ]:
col_names = [
    'checking_account',
    'duration_months',
    'credit_history',
    'purpose',
    'credit_amount',
    'savings_account',
    'employment_since',
    'installment_rate',
    'personal_status_sex',
    'other_debtors',
    'residence_since',
    'property',
    'age',
    'other_installment',
    'housing',
    'existing_credits',
    'job',
    'maintenance_people',
    'telephone',
    'foreign_worker',
    'target'
]

df = pd.read_csv(RAW_PATH, sep=' ', header=None, names=col_names)
print(f'Shape: {df.shape}')
df.head(3)

## 2. Recode Target

In [ ]:
df['target'] = (df['target'] == 2).astype(int)

print('Default rate:', df['target'].mean().round(4))
print(df['target'].value_counts())

## 3. Encode Ordered Features

In [ ]:

checking_account_map = {'A11': 0, 'A12': 1, 'A13': 2, 'A14': 3}
df['checking_account'] = df['checking_account'].map(checking_account_map)

credit_history_map = {
    'A30': 0,
    'A31': 1,
    'A32': 2,
    'A33': 3,
    'A34': 4,
}
df['credit_history'] = df['credit_history'].map(credit_history_map)

savings_account_map = {
    'A61': 0,
    'A62': 1,
    'A63': 2,
    'A64': 3,
    'A65': 4,
}
df['savings_account'] = df['savings_account'].map(savings_account_map)

employment_since_map = {
    'A71': 0,
    'A72': 1,
    'A73': 2,
    'A74': 3,
    'A75': 4,
}
df['employment_since'] = df['employment_since'].map(employment_since_map)

property_map = {
    'A121': 3,
    'A122': 2,
    'A123': 1,
    'A124': 0,
}
df['property'] = df['property'].map(property_map)

job_map = {
    'A171': 0,
    'A172': 1,
    'A173': 2,
    'A174': 3,
}
df['job'] = df['job'].map(job_map)

df['telephone'] = (df['telephone'] == 'A192').astype(int)

df['foreign_worker'] = (df['foreign_worker'] == 'A201').astype(int)

In [ ]:

nominal_cols = ['purpose', 'personal_status_sex', 'other_debtors',
                'other_installment', 'housing']

assert all(df[c].dtype == object or pd.api.types.is_string_dtype(df[c]) for c in nominal_cols), \
    'Nominal columns must stay as raw strings in the semi-raw export'
print('Nominal columns kept raw:', nominal_cols)
for c in nominal_cols:
    print(f'  {c}: {sorted(df[c].unique().tolist())}')
print(f'Shape: {df.shape}')

## 4. Sanity Checks

In [ ]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'None — clean.')

print('\nDtypes check:')
print(df.dtypes.value_counts())

print(f'\nTarget: {df["target"].sum()} defaults out of {len(df)} ({df["target"].mean():.1%})')

In [ ]:
num_cols = ['duration_months', 'credit_amount', 'installment_rate',
            'residence_since', 'age', 'existing_credits', 'maintenance_people']
df[num_cols].describe().round(2)

## 5. Save Processed Dataset

In [ ]:
df.insert(0, 'source_row_id', np.arange(len(df), dtype=np.int64))
cols = [c for c in df.columns if c != 'target'] + ['target']
df = df[cols]

df.to_csv(OUT_PATH, index=False)
print(f'Saved to {OUT_PATH}')
print(f'Final shape: {df.shape}')
df.head(3)

In [ ]:
import sys
sys.path.insert(0, '..')
from src.semiraw import audit_semiraw_export

report = audit_semiraw_export(OUT_PATH, 'south_german', expect_missing=False)